In [ ]:
# Module 10 — Code Along: the LLM as a task engine (support-ticket triage)
# SHARED DAY-4 BLOCK — domain + injected fake LLM. Paste once, runs everywhere (M10-M12).
from __future__ import annotations
import json
from types import SimpleNamespace
from typing import Optional, Any, Callable
from dataclasses import dataclass, field
from pydantic import BaseModel, Field, ValidationError

@dataclass
class SupportTicket:
    id: int
    subject: str
    category: str          # "Billing" | "Technical" | "Account"
    priority: str          # "low" | "normal" | "high" | "urgent"
    is_open: bool = True
    tags: list[str] = field(default_factory=list)

SAMPLE_TICKETS = [
    SupportTicket(1, "Can't log in after password reset", "Account", "high"),
    SupportTicket(2, "Invoice double-charged this month",  "Billing", "urgent"),
    SupportTicket(3, "How do I export my data?",            "Technical", "low"),
    SupportTicket(4, "App crashes on upload",               "Technical", "high"),
    SupportTicket(5, "Refund not received",                 "Billing", "normal", is_open=False),
]

# --- the dependency-injection seam: a fake stand-in for openai.OpenAI ---
def _msg(content=None, tool_calls=None):
    return SimpleNamespace(content=content, tool_calls=tool_calls)
def _resp(message):
    return SimpleNamespace(choices=[SimpleNamespace(message=message)])
def _tool_call(call_id, name, **args):
    return SimpleNamespace(id=call_id,
        function=SimpleNamespace(name=name, arguments=json.dumps(args)))

class FakeLLM:
    """Scripts chat.completions.create - same shape as openai.OpenAI, no API key.
    Exactly the Day-3 pattern: inject a fake at the seam instead of the real client."""
    def __init__(self, responses):
        self._responses = list(responses)
        self.chat = SimpleNamespace(
            completions=SimpleNamespace(create=self._create))
    def _create(self, **kwargs):
        return self._responses.pop(0)

print(f"{len(SAMPLE_TICKETS)} tickets loaded; FakeLLM ready (no API key needed)")

# Module 10 — The LLM is a task engine

An LLM is **text in, text out** — but pointed at a *task* it becomes a programmable function. Each task is spec'able: you write the instruction, fix the output shape, and validate what comes back.

| Task | What goes in | What comes out |
|---|---|---|
| **classify** | a ticket subject | a label (`priority`, `category`) |
| **extract** | natural-language request | a structured query object |
| **summarize** | a long ticket thread | one-line gist |
| **generate** | a few facts | a drafted reply |
| **transform** | text in form A | text in form B (translate, reformat) |
| **reason** | a question + data | a step-by-step answer |

This module covers **classify - extract - summarize**, then the **improvement ladder** (prompt -> few-shot -> RAG -> fine-tune). Every "LLM call" runs through the injected `FakeLLM` above — headless, no key.

In [ ]:
# TASK = CLASSIFY -- subject in, structured label out.
# Script the FakeLLM to answer with JSON (a real model returns text; we parse it).
llm = FakeLLM([
    _resp(_msg(content='{"priority": "urgent"}')),
])

subject = "Invoice double-charged this month"
prompt = f"Classify this support ticket's priority (low|normal|high|urgent). "\
         f"Reply as JSON {{\"priority\": ...}}. Subject: {subject!r}"

resp = llm.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
)
raw = resp.choices[0].message.content      # the model speaks text...
label = json.loads(raw)                     # ...we parse it into structure
print("raw   ->", raw)
print("label ->", label["priority"])        # urgent

## Structured output — make the LLM speak your schema

A free-text answer is hard to use; a **schema** turns the LLM into a typed function. Two levers:

- **Pydantic model** = the contract. You describe each field (and its `description=`) and the model becomes your parse target.
- **JSON mode** — pass `response_format={"type": "json_object"}` to the real client so the API guarantees valid JSON. (Our `FakeLLM` ignores it; mentioned so you recognise it in real code.)

The schema does double duty: it's the *instruction* to the model **and** the *validator* on the way back.

In [ ]:
# The extraction target: a Pydantic model. Every field's description doubles as
# the instruction we can hand the model (and JSON-mode honours the schema).
class TicketQuery(BaseModel):
    category: Optional[str] = Field(default=None, description="Restrict to this category, or null for all.")
    priority: Optional[str] = Field(default=None, description="One of low|normal|high|urgent, or null.")
    open_only: bool = False
    subject_contains: Optional[str] = Field(default=None, description="Substring the subject must contain.")

# model_json_schema() is exactly what you'd send as the function/JSON-mode schema.
print(json.dumps(TicketQuery.model_json_schema(), indent=2))

## Always validate — the LLM is just another untrusted source

Same boundary discipline as Day 2's HTTP bodies and CSV rows: **data crossing into your system is guilty until validated.** The LLM is no different — it can return the wrong shape, an extra field, or plain broken JSON.

So we never trust `message.content`. We feed it straight through `TicketQuery.model_validate_json(...)` and let Pydantic accept it or raise. Garbage in -> `ValidationError`, caught -> handled. Never a half-parsed dict leaking downstream.

In [ ]:
# TASK = EXTRACT + VALIDATE  (<- deck: module-10 cell 6)
# Natural language in, a validated TicketQuery out -- the same NL->structured task
# you build by hand in Lab 10's parse_nl_query.
llm = FakeLLM([
    # 1) a well-formed extraction
    _resp(_msg(content='{"category": "Billing", "open_only": true}')),
    # 2) a malformed reply (truncated JSON) -- the LLM misbehaving
    _resp(_msg(content='{"category": "Billing", "open_only":')),
])

def extract_query(user_text: str) -> TicketQuery:
    resp = llm.chat.completions.create(
        model="gpt-4o-mini",
        response_format={"type": "json_object"},   # real client honours this; FakeLLM ignores it
        messages=[{"role": "user", "content": f"Extract a TicketQuery from: {user_text!r}"}],
    )
    raw = resp.choices[0].message.content
    return TicketQuery.model_validate_json(raw)     # validate at the boundary

# Good case: parses into a typed object
q = extract_query("show me open billing tickets")
print("OK ->", q)

# Bad case: malformed JSON is caught, not crashed
try:
    extract_query("anything")
except ValidationError as e:
    print("ValidationError caught:", e.error_count(), "error(s) -- we never trusted it")

## TASK = summarize — collapse a thread to its gist

A support thread can be dozens of turns. Summarize compresses it to one actionable line an agent can scan. Same pattern: thread in, short text out — here we keep the output as plain text (no schema needed).

In [ ]:
# TASK = SUMMARIZE -- a multi-turn thread in, one line out.
thread = [
    "Customer: I was charged twice for my June invoice.",
    "Agent: I see two charges of $49 on the 3rd. Let me check.",
    "Customer: Yes, please refund the duplicate.",
    "Agent: Refund issued, 3-5 business days.",
]

llm = FakeLLM([
    _resp(_msg(content="Duplicate $49 June charge refunded; ETA 3-5 business days.")),
])

resp = llm.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user",
               "content": "Summarise this support thread in one line:\n" + "\n".join(thread)}],
)
print("SUMMARY:", resp.choices[0].message.content)

## The improvement ladder — climb only as far as the task needs

| Rung | Lever | Reach for it when |
|---|---|---|
| **1. Prompt** | clearer instruction + schema | first stop, always — most tasks stop here |
| **2. Few-shot** | 2-5 examples in the prompt | format is fiddly; one example teaches the shape |
| **3. RAG** | retrieve facts, inject into prompt | the model **lacks knowledge** (your docs, live data) |
| **4. Fine-tune** | train on many (input, output) pairs | high-volume, **narrow**, **fixed format**; cut cost/latency |

**Fine-tune is worth it when:** the task is repetitive, the output shape is rigid, and you have hundreds+ examples — it bakes the behaviour in so prompts shrink (cheaper, faster, more consistent).

**Don't fine-tune when:** you need *knowledge* (use RAG) or the target is a *moving target* (you'd retrain forever). Fine-tuning teaches **behaviour/format**, not facts.

In [ ]:
# FINE-TUNING DATA PREP (shown) -- build training rows in memory as JSONL.
# Task = NL subject -> structured TicketQuery JSON: the SAME task as Lab 10's parse_nl_query.
SYSTEM = "Extract a TicketQuery JSON from the user's support request."

def to_example(t: SupportTicket) -> dict:
    # the "label" we'd want the tuned model to emit for this subject
    target = TicketQuery(category=t.category, priority=t.priority).model_dump()
    return {"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": t.subject},
        {"role": "assistant", "content": json.dumps(target)},
    ]}

dataset = [to_example(t) for t in SAMPLE_TICKETS]

# tickets.jsonl = one JSON object per line (what the fine-tuning API ingests)
jsonl = "\n".join(json.dumps(ex) for ex in dataset)
print(f"{len(dataset)} training rows\n")
print("--- one example, pretty-printed ---")
print(json.dumps(dataset[0], indent=2))

## The fine-tuning job (shown, not run)

In real life you'd upload `tickets.jsonl`, kick off a job, and get back a new model id. We **don't run it** here (it costs money and hits the network) — but the call is one line. What matters more is the rung below it: **you cannot tell if tuning helped without an eval.**

In [ ]:
# The fine-tuning job is ONE call -- kept commented so nothing hits the network:
# client.fine_tuning.jobs.create(training_file=file_id, model="gpt-4o-mini")
#   -> returns a job; when done you get a new model id like "ft:gpt-4o-mini:...".

# What actually tells you if tuning helped: a HELD-OUT eval. Tiny harness below,
# fully runnable with plain-Python stand-ins for the base vs tuned model.
HELD_OUT = [
    ("Invoice double-charged this month", "urgent"),
    ("App crashes on upload",             "high"),
    ("How do I export my data?",          "low"),
    ("Refund not received",               "normal"),
]

def score(predict_fn, cases) -> float:
    correct = sum(predict_fn(subject) == expected for subject, expected in cases)
    return correct / len(cases)

# "base" model: weak, defaults everything to "normal"
def base_predict(subject: str) -> str:
    return "normal"

# "tuned" model: learned the mapping from the training rows
def tuned_predict(subject: str) -> str:
    for t in SAMPLE_TICKETS:
        if t.subject == subject:
            return t.priority
    return "normal"

print(f"base  accuracy: {score(base_predict,  HELD_OUT):.0%}")
print(f"tuned accuracy: {score(tuned_predict, HELD_OUT):.0%}")   # tuned scores higher

## Recap

- The LLM is a **task engine**: classify - extract - summarize - generate - transform - reason. Each is spec'able.
- **Structured output** = make it speak a Pydantic schema; **JSON mode** guarantees parseable JSON.
- **Validate at the boundary** — the LLM is just another untrusted source (`model_validate_json` -> catch `ValidationError`).
- **Improvement ladder:** prompt -> few-shot -> RAG -> fine-tune. Climb only as far as the task needs; tune for *format*, RAG for *knowledge*.
- A change is only an improvement if a **held-out eval** says so — that's exactly the golden-eval you build in **Module 12**.

**-> Lab 10 applies this same NL->structured pattern to Product / `CatalogQuery`.**